Part 3

ATTACK 1: EVASION ATTACK

Character-level perturbation function

In [ ]:
import random
import re

ZERO_WIDTH = "\u200b"

homoglyphs = {
    "a": "а",
    "e": "е",
    "o": "о",
    "i": "і",
    "c": "с"
}

def perturb(text):
    words = text.split()

    new_words = []

    for word in words:
        chars = list(word)
        new_chars = []

        for i, ch in enumerate(chars):

            # 1. homoglyph substitution
            if ch.lower() in homoglyphs and random.random() < 0.5:
                ch = homoglyphs[ch.lower()]

            new_chars.append(ch)

            # 2. random duplication (20%)
            if random.random() < 0.2:
                new_chars.append(ch)

            # 3. zero-width space insertion
            if i % 2 == 0 and random.random() < 0.3:
                new_chars.append(ZERO_WIDTH)

        new_words.append("".join(new_chars))

    return " ".join(new_words)

Select toxic high-confidence samples

In [ ]:
import numpy as np
from scipy.special import softmax

pred_output = trainer.predict(eval_dataset)

logits = pred_output.predictions
labels = pred_output.label_ids

probs = softmax(logits, axis=1)[:, 1]

eval_df = eval_dataset.to_pandas()
eval_df["label"] = labels
eval_df["prob"] = probs

# only toxic + high confidence
attack_samples = eval_df[
    (eval_df["label"] == 1) &
    (eval_df["prob"] >= 0.7)
].sample(500, random_state=42).copy()

Apply perturbation

In [ ]:
attack_samples["perturbed"] = attack_samples["comment_text"].apply(perturb)


Load model for inference

In [ ]:
from transformers import pipeline

clf = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    return_all_scores=True
)

Get probability function

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def get_prob(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    # 🔥 MOVE INPUTS TO SAME DEVICE AS MOD

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)

    return probs[0][1].item()

In [ ]:
attack_samples["old_prob"] = attack_samples["prob"]

attack_samples["new_prob"] = attack_samples["perturbed"].apply(get_prob)

In [ ]:
threshold = 0.3  # your chosen threshold

attack_samples["new_pred"] = (attack_samples["new_prob"] >= threshold).astype(int)

In [ ]:
ASR = (attack_samples["new_pred"] == 0).mean()

print("===== EVASION ATTACK RESULTS =====")
print("Attack Success Rate (ASR):", ASR)
print("Avg confidence BEFORE:", attack_samples["old_prob"].mean())
print("Avg confidence AFTER :", attack_samples["new_prob"].mean())

===== EVASION ATTACK RESULTS =====
Attack Success Rate (ASR): 0.96
Avg confidence BEFORE: 0.92792946
Avg confidence AFTER : 0.0405569650423713


Second Attack: Poisoning

Create Poisoned Training Set

In [ ]:
import pandas as pd

# Copy training subset (100k rows assumed)
train_poisoned = train_df.copy()

# Select 5% of data
poison_size = int(0.05 * len(train_poisoned))
poison_idx = train_poisoned.sample(n=poison_size, random_state=42).index

# Flip labels: 0 ↔ 1
train_poisoned.loc[poison_idx, "label"] = 1 - train_poisoned.loc[poison_idx, "label"]

print("Poisoned samples:", poison_size)
print(train_poisoned["label"].value_counts())

Poisoned samples: 5000
label
0    87823
1    12177
Name: count, dtype: int64


Convert to HuggingFace Dataset

In [ ]:
from datasets import Dataset

poison_dataset = Dataset.from_pandas(train_poisoned)
eval_dataset_tok = Dataset.from_pandas(eval_df)

Tokenization

In [ ]:
def tokenize_function(batch):
    return tokenizer(
        batch["comment_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

poison_dataset = poison_dataset.map(tokenize_function, batched=True)
eval_dataset_tok = eval_dataset_tok.map(tokenize_function, batched=True)

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Rename Label Column

In [ ]:
poison_dataset = poison_dataset.rename_column("label", "labels")
eval_dataset_tok = eval_dataset_tok.rename_column("label", "labels")

Remove Extra Columns

In [ ]:
columns_to_keep = ["input_ids", "attention_mask", "labels"]

poison_dataset = poison_dataset.remove_columns(
    [col for col in poison_dataset.column_names if col not in columns_to_keep]
)

eval_dataset_tok = eval_dataset_tok.remove_columns(
    [col for col in eval_dataset_tok.column_names if col not in columns_to_keep]
)

Ensure Label Type is Correct

In [ ]:
def fix_labels(example):
    example["labels"] = int(example["labels"])
    return example

poison_dataset = poison_dataset.map(fix_labels)
eval_dataset_tok = eval_dataset_tok.map(fix_labels)

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Load Fresh DistilBERT Model

In [ ]:
from transformers import DistilBertForSequenceClassification

poison_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Metrics Function

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "f1": float(f1_score(labels, preds))
    }

Data Collator

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Training Arguments

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert-poisoned",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=200,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    learning_rate=2e-5,
    report_to=[],
    load_best_model_at_end=False
)

Trainer Setup

In [ ]:
from accelerate.state import AcceleratorState

AcceleratorState._reset_state()

In [ ]:
poison_trainer = Trainer(
    model=poison_model,
    args=training_args,
    train_dataset=poison_dataset,
    eval_dataset=eval_dataset_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

Train Poisoned Model

In [ ]:
poison_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.304532,0.159811,0.947800,0.608696
2,0.284501,0.160806,0.946600,0.619387
3,0.201428,0.193987,0.940100,0.618471


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=18750, training_loss=0.264917660929362, metrics={'train_runtime': 4016.0827, 'train_samples_per_second': 74.7, 'train_steps_per_second': 4.669, 'total_flos': 9935054899200000.0, 'train_loss': 0.264917660929362, 'epoch': 3.0})

Evaluate Poisoned Model

In [ ]:
poison_results = poison_trainer.evaluate()
print("Poisoned Model Results:", poison_results)

Poisoned Model Results: {'eval_loss': 0.19398730993270874, 'eval_accuracy': 0.9401, 'eval_f1': 0.6184713375796178, 'eval_runtime': 87.0928, 'eval_samples_per_second': 229.64, 'eval_steps_per_second': 14.353, 'epoch': 3.0}


Evaluate Clean Model

In [ ]:
clean_results = trainer.evaluate()

print("\n=== COMPARISON ===")
print("Clean Model:", clean_results)
print("Poisoned Model:", poison_results)


=== COMPARISON ===
Clean Model: {'eval_loss': 0.15267208218574524, 'eval_accuracy': 0.9489, 'eval_f1_macro': 0.7930530291285759, 'eval_auc': 0.9482312239399936, 'eval_runtime': 89.3038, 'eval_samples_per_second': 223.955, 'eval_steps_per_second': 13.997, 'epoch': 3.0}
Poisoned Model: {'eval_loss': 0.19398730993270874, 'eval_accuracy': 0.9401, 'eval_f1': 0.6184713375796178, 'eval_runtime': 87.0928, 'eval_samples_per_second': 229.64, 'eval_steps_per_second': 14.353, 'epoch': 3.0}


False Negative Rate

In [ ]:
from sklearn.metrics import confusion_matrix

preds = poison_trainer.predict(eval_dataset_tok)

y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

fnr = fn / (fn + tp)

print("False Negative Rate (Poisoned Model):", fnr)

False Negative Rate (Poisoned Model): 0.39274546591619763


## Threat Model Comparison: Evasion vs. Poisoning (Part 3)

### Attack Results Summary

| Attack Type | Success Metric | Result |
|-------------|----------------|--------|
| Evasion (Character-level perturbations) | Attack Success Rate (ASR) | **96.0%** |
| Evasion | Avg confidence drop | 0.928 → 0.041 |
| Poisoning (5% label flipping) | F1 Score (clean → poisoned) | 0.793 → 0.618 |
| Poisoning | Accuracy drop | 0.949 → 0.940 |

### Which attack is more operationally dangerous?

**The poisoning attack is more operationally dangerous** for a live platform, despite the evasion attack having a higher numerical success rate.

### Why poisoning is more dangerous:

1. **Persistence:** A poisoned model remains corrupted for all users until retrained. A single successful poisoning event can affect millions of predictions over weeks or months. Evasion attacks must be executed per comment and affect only that specific input.

2. **Detection difficulty:** Poisoning happens silently during training. By the time performance degradation is noticed, the model may have already been serving incorrect predictions for an extended period. Evasion attacks produce obvious perturbations (zero-width characters, homoglyphs) that can be detected via input sanitization.

3. **Lateral damage:** Poisoning affects the entire user base indiscriminately. Evasion attacks target specific comments from specific attackers.

4. **Recovery cost:** Fixing a poisoned model requires full retraining with verified clean data and potentially forensic analysis to identify the poisoned samples. Evasion can be mitigated with input preprocessing or adversarial training.

### Which threat model is more realistic for a social platform?

**Both are realistic, but poisoning is more plausible for sophisticated adversaries.**

| Threat Model | Realism | Reasoning |
|--------------|---------|-----------|
| Evasion | Medium | Requires per-comment effort. Automated scripts can generate perturbations, but perturbed text is often detectable (unusual characters, duplicate letters). Platforms can implement input normalization. |
| Poisoning | High | Many platforms rely on user reports or crowd-sourced moderation to generate training labels. An organized group could systematically submit false reports to flip labels on specific comments, injecting poisoned data into the next training batch. |

**Realistic poisoning scenario:** A coordinated adversarial group identifies that the platform retrains weekly using reported comments. They submit hundreds of non-toxic comments with false "toxic" reports, and toxic comments with false "not toxic" reports. Over several weeks, the model's decision boundary shifts.

### Implications for defense prioritization

**Defenses should prioritize training pipeline integrity over input filtering.**

| Priority | Defense | Rationale |
|----------|---------|-----------|
| **Highest** | Training data validation | Detect label flipping anomalies, outlier detection on training samples, robust aggregation of crowd-sourced labels |
| **High** | Regular model retraining with verified holdout sets | Detect performance degradation early |
| **Medium** | Input sanitization | Remove zero-width characters, detect homoglyphs, limit repeated characters |
| **Medium** | Adversarial training | Inject perturbed examples during training to improve robustness |
| **Low** | Post-hoc detection | Evasion is visible; reactive blocking is possible but not preventative |

### Conclusion

While the evasion attack achieved an alarming 96% success rate in our controlled experiment, a real platform would likely detect and filter such obvious perturbations. The poisoning attack, despite showing smaller numerical degradation in our 5% flip experiment, represents a more insidious and harder-to-detect threat that compromises the model's integrity at its foundation. **Platforms should invest more heavily in training pipeline security and data validation than in input sanitization.**